In [9]:
import os
import numpy as np
import warnings 
from string import Template
import xarray as xr
import numpy as np
import pandas as pd

from scipy.stats import linregress

from datetime import datetime

from ObservationDataUtil import ( 
     ObsDataReader,
    
)
from ModelDataUtil import (
    ModelDataReader,
)

from experiment_configs import get_experiment_dict


In [10]:
class ObservedTCICalculator:
    def __init__(self, obs_dict, sm_source, flux_source, sm_var, flux_var,
                 years, outdir='./obs_tci_output',
                 lat_bounds=None, lon_bounds=None, target_depth_m=None):
        self.obs_dict = obs_dict
        self.sm_source = sm_source
        self.flux_source = flux_source
        self.sm_var = sm_var
        self.flux_var = flux_var
        self.years = years
        self.outdir = outdir
        os.makedirs(outdir, exist_ok=True)

        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.target_depth_m = target_depth_m
        self.Lv = 2.5e6  # J/kg

        self.regions = {
            "CONUS": {"lat": (25, 50), "lon": (-125, -65)},
            "SouthAmerica": {"lat": (-55, 15), "lon": (-80, -30)},
            "Australia": {"lat": (-45, -10), "lon": (110, 155)},
            "EastAsia": {"lat": (20, 50), "lon": (100, 140)},
            "Africa": {"lat": (-35, 35), "lon": (-20, 50)},
        }

    def normalize_longitude_and_sort(self, ds, lon_name='lon'):
        if lon_name in ds.coords:
            lon = ds[lon_name]
            if lon.max() > 180:
                ds[lon_name] = ((lon + 180) % 360) - 180
                ds = ds.sortby(lon_name)
        return ds

    def _resample_if_needed(self, ds, frequency):
        if frequency == '6hourly':
            ds_daily = ds.resample(time='1D').mean()
            ds_daily['time'] = pd.to_datetime(ds_daily['time'].values).normalize()
            return ds_daily
        return ds

    def _load_obs_variable(self, source, varname):
        info = self.obs_dict[source]
        base_path = info['path']
        template = info['template']
        frequency = info.get('frequency', 'daily')

        files = [
            os.path.join(base_path, template % {"var": varname, "year": y})
            for y in self.years
        ]
        if not all(os.path.exists(f) for f in files):
            raise FileNotFoundError(f"[ERROR] Missing files for {source} - {varname}")

        da_list = [xr.open_dataset(f)[varname] for f in files]
        da = xr.concat(da_list, dim='time')
        da = self._resample_if_needed(da, frequency)

        if 'latitude' in da.coords:
            da = da.rename({'latitude': 'lat'})
        if 'longitude' in da.coords:
            da = da.rename({'longitude': 'lon'})

        da = self.normalize_longitude_and_sort(da)

        if self.lat_bounds:
            da = da.sel(lat=slice(*self.lat_bounds))
        if self.lon_bounds:
            da = da.sel(lon=slice(*self.lon_bounds))

        if varname == 'QFLX':
            da = da * self.Lv
            da.attrs["units"] = "W/m2"
            da.name = "LHFLX"
            
        if varname == 'H2OSOI':
            da = da * 0.05 * 1000.0 # convert to kg/m2 
            da.attrs["units"] = "kg/m2"
            da.name = "H2OSOI"

        return da

    def _compute_tci_dirmeyer(self, sm, flux):
        common_time = np.intersect1d(sm['time'], flux['time'])
        if len(common_time) < 2:
            raise ValueError("[ERROR] Not enough overlapping time steps.")

        sm = sm.sel(time=common_time)
        flux = flux.sel(time=common_time)

        sm_stack = sm.stack(grid=('lat', 'lon')).transpose('time', 'grid')
        flux_stack = flux.stack(grid=('lat', 'lon')).transpose('time', 'grid')

        tci_vals = []
        for i in range(sm_stack.shape[1]):
            x = sm_stack[:, i].values
            y = flux_stack[:, i].values
            if np.all(np.isfinite(x)) and np.all(np.isfinite(y)) and not np.allclose(x, x[0]):
                slope, _, _, _, _ = linregress(x, y)
                tci_vals.append(slope * np.std(x))
            else:
                tci_vals.append(np.nan)

        lat = sm['lat']
        lon = sm['lon']
        tci_array = np.array(tci_vals).reshape(len(lat), len(lon))
        return xr.DataArray(tci_array, coords={'lat': lat, 'lon': lon}, dims=['lat', 'lon'], name='TCI')

    def compute_weighted_mean(self, da):
        weights = np.cos(np.deg2rad(da['lat']))
        weights.name = "weights"
        return da.weighted(weights).mean(dim=['lat', 'lon'], skipna=True)

    def save_regional_mean(self, tci, tag):
        data_vars = {}
        for region, bounds in self.regions.items():
            if bounds['lon'][0] < bounds['lon'][1]:
                tci_region = tci.sel(
                    lat=slice(*bounds['lat']),
                    lon=slice(*bounds['lon'])
                )
            else:
                west = tci.sel(lon=slice(bounds['lon'][0], 180))
                east = tci.sel(lon=slice(-180, bounds['lon'][1]))
                tci_region = xr.concat([west, east], dim='lon')

            mean_val = self.compute_weighted_mean(tci_region)
            data_vars[f"TCI_{region}"] = mean_val

        ds_out = xr.Dataset(data_vars)
        outfile = os.path.join(self.outdir, f"TCI_obs_regional_mean_{tag}.nc")
        ds_out.to_netcdf(outfile)
        print(f"[SAVED] Regional mean: {outfile}")

    def process(self, windows_dict, compute_regional_mean=True):
        print("[INFO] Loading observational variables...")
        sm = self._load_obs_variable(self.sm_source, self.sm_var)
        flux = self._load_obs_variable(self.flux_source, self.flux_var)

        for tag, (start_str, end_str) in windows_dict.items():
            print(f"[INFO] Processing window {tag}")
            start = np.datetime64(start_str)
            end = np.datetime64(end_str)

            sm_window = sm.sel(time=slice(start, end))
            flux_window = flux.sel(time=slice(start, end))

            if sm_window.time.size < 2:
                print(f"[WARN] Skipping window {tag} due to insufficient data")
                continue

            tci = self._compute_tci_dirmeyer(sm_window, flux_window)
            tci['time'] = sm_window.time.mean()
            tci = tci.expand_dims('time')

            filename = f"TCI_obs_window_{tag}.nc"
            outfile = os.path.join(self.outdir, filename)
            tci.to_netcdf(outfile)
            print(f"[SAVED] {outfile}")

            if compute_regional_mean:
                self.save_regional_mean(tci, tag)


In [14]:
if __name__ == "__main__":
    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = f"{top_path}/diag_dart"
    fig_path = "/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    landmask_file = f"{out_path}/landmask_1x1.nc"
    soilayer_file = f"{out_path}/dzsoi_elm.nc"

    obs_dict = {
        'ERA5': {
            'path': f'{top_path}/Observations/ERA5/6hourly',
            'template': 'ERA5.6hourly.en00.%(var)s.%(year)s01-%(year)s12.nc',
            'nens': 1
        },
        'ESA_CCI': {
            'path': f'{top_path}/Observations/ESA_CCI/daily',
            'template': 'H2OSOI.daily.%(year)s.nc',
            'frequency': 'daily',
            'nens': 1
        },
    }
    
    # Labeled time windows
    years = [2011]   
    windows_dict = {
        "D1-15-2011":   ("2011-12-01", "2011-12-15"),
        "D16-30-2011":  ("2011-12-16", "2011-12-31"),
        "2011-12": ("2011-12-01", "2011-12-31"),
    }    
    
    years = [2012]   
    windows_dict = {
        "D1-15":   ("2012-01-01", "2012-01-15"),
        "D16-30":  ("2012-01-16", "2012-01-30"),
        "D31-45":  ("2012-01-31", "2012-02-14"),
        "D46-60":  ("2012-02-15", "2012-03-01"),
        "2012-01": ("2012-01-01", "2012-01-31"),
        "2012-02": ("2012-02-01", "2012-02-29"),
    }
    
    # --- Observation TCI Processing ---
    print(f"\n[INFO] Running TCI for Observations (ESA_CCI + ERA5)")

    obs_tci_calc = ObservedTCICalculator(
        obs_dict=obs_dict,
        sm_source='ESA_CCI',
        flux_source='ERA5',
        sm_var='H2OSOI',
        flux_var='LHFLX',
        years=years,
        outdir=os.path.join(out_path, "obs_tci"),
        target_depth_m=None  # surface-only data, no integration
    )

    obs_tci_calc.process(windows_dict=windows_dict, compute_regional_mean=True)



[INFO] Running TCI for Observations (ESA_CCI + ERA5)
[INFO] Loading observational variables...
[INFO] Processing window D1-15
[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_window_D1-15.nc
[SAVED] Regional mean: /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_regional_mean_D1-15.nc
[INFO] Processing window D16-30
[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_window_D16-30.nc
[SAVED] Regional mean: /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_regional_mean_D16-30.nc
[INFO] Processing window D31-45
[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_window_D31-45.nc
[SAVED] Regional mean: /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_regional_mean_D31-45.nc
[INFO] Processing window D46-60
[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_window_D46-60.nc
[SAVED] Regional mean: /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/obs_tci/TCI_obs_regional_mean_D46-60.nc
[INFO] Processing w